# 03 — Reranking

Stack a cross-encoder (`bge-reranker-v2-m3`) on top of hybrid and measure ΔnDCG / ΔPrecision@1.


In [ ]:
import json
from rag_evals.config import settings
from rag_evals.evaluation.retrieval import evaluate_runs, precision_at_k
from rag_evals.retrieval.dense import DenseRetriever
from rag_evals.retrieval.sparse import SparseRetriever
from rag_evals.retrieval.hybrid_rrf import HybridRetriever
from rag_evals.retrieval.reranker import CrossEncoderReranker

rows = [json.loads(l) for l in (settings.golden_dir / "retrieval.jsonl").open()][:30]
hybrid = HybridRetriever(DenseRetriever(), SparseRetriever(), k=60)
rerank = CrossEncoderReranker()

before, after, gold = {}, {}, {}
for r in rows:
    hits = hybrid(r["query"], limit=20, per_lane=50)
    before[r["qid"]] = [h.doc_id for h in hits]
    after[r["qid"]]  = [h.doc_id for h in rerank(r["query"], hits, limit=10)]
    gold[r["qid"]]   = r["gold_doc_ids"]

before_m = evaluate_runs(before, gold, k=10)
after_m  = evaluate_runs(after, gold, k=10)
print(f"before rerank: nDCG@10={before_m.ndcg_at_k:.3f} P@1={before_m.precision_at_k:.3f}")
print(f" after rerank: nDCG@10={after_m.ndcg_at_k:.3f} P@1={after_m.precision_at_k:.3f}")
print(f"   ΔnDCG@10 = {after_m.ndcg_at_k - before_m.ndcg_at_k:+.3f}")
